In [27]:
print(test_haversine( 30, 40, 10,20,))
print(test_haversine( 10,20, 30, 40))
testA = np.array([1,5,6,2,3])
testB = np.array([7,7,1,1,1])
testC = np.array([-1,8,2,3,4])
testD = np.sum([testA, testB, testC], axis = 0)
print(testD)

import numpy as np


testE = np.minimum.reduce([testA, testB, testC])
# print(result)
print(testE)
print(np.minimum(testE, np.full(5, fill_value = 3)))
testF = 1 - 4 / np.minimum(testE, np.full(5, fill_value = 3))
print(testF)
# testE = 1 - testB / np.maximum(testE, np.full(5, fill_value = 3))

3040.6028180682006
3040.6028180682006
[ 7 20  9  6  8]
[-1  5  1  1  1]
[-1  3  1  1  1]
[ 5.         -0.33333333 -3.         -3.         -3.        ]


In [55]:
import numpy as np

a = np.array([0, 0, 0, 2, 2, 2])
b = np.array([0, 0.7, 0.7, 1, 1, 0])

# 同时满足两个条件的索引
indices = np.where( (a > 1) & (b > 0.5) )[0]

print(indices)  # 输出: [3 4]

[3 4]


In [ ]:
import numpy as np

def test_haversine(lat1, lon1, lat2, lon2):
    r = 6371000
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    
    sin_dlat = np.sin(dlat / 2)
    sin_dlon = np.sin(dlon / 2)
    
    rad_lat1 = np.radians(lat1)
    rad_lat2 = np.radians(lat2)
    
    a = sin_dlat * sin_dlat + np.cos(rad_lat1) * np.cos(rad_lat2) * sin_dlon * sin_dlon
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    d = r * c
    return d 

def compute_sim(ids, sender_aois, embeddings, sender_lat, sender_lng, rec_lat, rec_lng):
    triu_i, triu_j = np.triu_indices(n = len(embeddings), k = 1)
    
    sender_aoi_map = dict()
    sender_aoi_i = sender_aois[triu_i]
    sender_aoi_j = sender_aois[triu_j]
    
    embd_1 = embeddings[triu_i]
    embd_2 = embeddings[triu_j]
    
    cos_sim = compute_cosine_sim(embd_1, embd_2)
    
    sender_lat = sender_lat / 1e6
    sender_lng = sender_lng / 1e6
    rec_lat = rec_lat / 1e6
    rec_lng = rec_lng / 1e6
    
    s_lat1 = sender_lat[triu_i]
    s_lng1 = sender_lng[triu_i]
    r_lat1 = rec_lat[triu_i]
    r_lng1 = rec_lng[triu_i]
    
    s_lat2 = sender_lat[triu_j]
    s_lng2 = sender_lng[triu_j]
    r_lat2 = rec_lat[triu_j]
    r_lng2 = rec_lng[triu_j]
    
    share_degree = compute_edges_new(
        s_lat1,
        s_lng1,
        r_lat1,
        r_lng1,
        s_lat2,
        s_lng2,
        r_lat2,
        r_lng2
    )
    
    # pass
    print(cos_sim[:10])
    print("==========")
    print(share_degree[:10])

def compute_cosine_sim(embd_1, embd_2):
    embd_products = np.sum(embd_1 * embd_2, axis=1)
    norm_1 = np.linalg.norm(embd_1, axis=1)
    norm_2 = np.linalg.norm(embd_2, axis=1)
    denominator = norm_1 * norm_2
    epsilon = 1e-8
    cos_sim = embd_products / (denominator + epsilon)
    return cos_sim

def compute_edges_new(
    s_lat1 = None, 
    s_lng1 = None, 
    r_lat1 = None,
    r_lng1 = None,
    s_lat2 = None, 
    s_lng2 = None, 
    r_lat2 = None, 
    r_lng2 = None):

    dist_01 = test_haversine(s_lat1, s_lng1, r_lat1, r_lng1)
    dist_02 = test_haversine(s_lat1, s_lng1, s_lat2, s_lng2)
    dist_03 = test_haversine(s_lat1, s_lng1, r_lat2, r_lng2)
    dist_12 = test_haversine(r_lat1, r_lng1, s_lat2, s_lng2)
    dist_13 = test_haversine(r_lat1, r_lng1, r_lat2, r_lng2)
    dist_23 = test_haversine(s_lat2, s_lng2, r_lat2, r_lng2)
    
    dist_order_1 = dist_01 + dist_12 + dist_23
    dist_order_2 = dist_02 + dist_12 + dist_13
    dist_order_3 = dist_02 + dist_23 + dist_13
    dist_order_4 = dist_02 + dist_01 + dist_13
    dist_order_5 = dist_02 + dist_03 + dist_13
    dist_order_6 = dist_23 + dist_03 + dist_01
    
    min_dist = np.minimum.reduce([dist_order_1, dist_order_2, dist_order_3,
                                  dist_order_4, dist_order_5, dist_order_6])
    sep_dist = dist_01 + dist_23
    share_degree = 1 - min_dist / np.maximum(sep_dist, np.full(s_lat1.shape[0], fill_value = 1e-9))
    return share_degree

if __name__ == "__main__":
    N = 10
    np.random.seed(2025)
    embeddings = np.random.randn(N, 5)  # 示例数据
    # embeddings = embeddings.tolist()
    # print(embeddings)
    ids = [f"id_{i}" for i in range(N)],
    sender_aois = [f'aoi_{i}' for i in range(N)],
    coords = np.random.randn(N, 4)
    sender_lat = coords[:, 0]
    sender_lng = coords[:, 1]
    rec_lat = coords[:, 2]
    rec_lng = coords[:, 3]
    compute_sim(ids, sender_aois, embeddings, sender_lat, sender_lng, rec_lat, rec_lng)
    
    # data = {
    #     'ids': [f"id_{i}" for i in range(N)],
    #     "sender_aois": [f'aoi_{i}' for i in range(N)],
    #     'embeddings': embeddings,
    #     'sender_lat': sender_lat,
    #     'sender_lng': sender_lng,
    #     'recipient_lat': rec_lat, 
    #     'recipient_lng': rec_lng
    # }
    # origin(data, N)
    # np.random.seed(19)
    # embeddings = np.random.randn(10, 5)  # 示例数据
    # coords = np.random.randn
    # cos_sim = compute_cosine_sim(embeddings=embeddings)
    # print(cos_sim[:10])

[ 0.12666092  0.35029603  0.27937887 -0.16193558 -0.35482159 -0.10335807
  0.39430733 -0.65166129 -0.0581742  -0.69918932]
[-0.5280111  -0.93655094 -0.37271391 -0.32063533 -0.56341368 -0.31145636
 -0.39584587 -0.08465502 -0.72456838 -0.60074657]


In [53]:
from multiprocessing import Pool, cpu_count
from itertools import combinations
from sklearn.metrics.pairwise import cosine_similarity
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

def safe_cosine_similarity(a, b):
    if a is None or b is None or len(a) == 0 or len(b) == 0 or len(a) != len(b):
        return 0
    return cosine_similarity([a], [b])[0][0]

# 向量化Haversine距离计算
def haversine(lat1, lng1, lat2, lng2):
    lat1, lng1 = np.radians(lat1/1e6), np.radians(lng1/1e6)
    lat2, lng2 = np.radians(lat2/1e6), np.radians(lng2/1e6)
    dlat, dlng = lat2 - lat1, lng2 - lng1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlng/2)**2
    return 6371000 * 2 * np.arcsin(np.sqrt(a))

# 计算共享距离
def share_degree_calculation(flow_coordinates_1, flow_coordinates_2):
    if 0 in flow_coordinates_1 or 0 in flow_coordinates_2:
        return -1
    wb1_sender = (flow_coordinates_1[0], flow_coordinates_1[1])
    wb1_rec = (flow_coordinates_1[2], flow_coordinates_1[3])
    wb2_sender = (flow_coordinates_2[0], flow_coordinates_2[1])
    wb2_rec = (flow_coordinates_2[2], flow_coordinates_2[3])

    points = [wb1_sender, wb1_rec, wb2_sender, wb2_rec]
    orders = [[0, 1, 2, 3], [0, 2, 1, 3], [0, 2, 3, 1], [2, 0, 1, 3], [2, 0, 3, 1], [2, 3, 0, 1]]
    min_distance = float('inf')

    # 预先计算 Haversine 距离并存储到字典
    distance_cache = {}
    for i in range(len(points)):
        for j in range(i + 1, len(points)):
            
            point1 = points[i]
            point2 = points[j]
            distance_cache[tuple(sorted((point1, point2)))] = haversine(point1[0], point1[1], point2[0], point2[1])

    for order in orders:
        distance = 0
        for i in range(len(order) - 1):
            point1 = points[order[i]]
            point2 = points[order[i + 1]]
            # 从字典中查找预先计算的距离
            distance += distance_cache[tuple(sorted((point1, point2)))]
        min_distance = min(min_distance, distance)
    sep_distance = (distance_cache[tuple(sorted((wb1_sender, wb1_rec)))] +
                    distance_cache[tuple(sorted((wb2_sender, wb2_rec)))])
    
    return 1 - min_distance / max(sep_distance, 1e-9)

def origin(data, n):
    sims = []
    shared = []
    for i in range(n):
        for j in range(i + 1, n):
            # if data['sender_aois'][i] != data['sender_aois'][j]:
            #     return None
            emb_i = data['embeddings'][i]
            emb_j = data['embeddings'][j]
            sim = safe_cosine_similarity(emb_i, emb_j)
            coords_i = (
                data['sender_lat'][i], data['sender_lng'][i],
                data['recipient_lat'][i], data['recipient_lng'][i]
            )
            coords_j = (
                data['sender_lat'][j], data['sender_lng'][j],
                data['recipient_lat'][j], data['recipient_lng'][j]
            )
            share_deg = share_degree_calculation(coords_i, coords_j)
            # print(i, j, sim, share_deg)
            sims.append(sim)
            shared.append(share_deg)
    print(sims )
    print("========")
    print(shared)
    
if __name__ == "__main__":
    N = 10
    np.random.seed(2025)
    embeddings = np.random.randn(N, 5)  # 示例数据
    # embeddings = embeddings.tolist()
    # print(embeddings)
    coords = np.random.randn(N, 4)
    sender_lat = coords[:, 0]
    sender_lng = coords[:, 1]
    rec_lat = coords[:, 2]
    rec_lng = coords[:, 3]
    data = {
        'ids': [f"id_{i}" for i in range(N)],
        "sender_aois": [f'aoi_{i}' for i in range(N)],
        'embeddings': embeddings,
        'sender_lat': sender_lat,
        'sender_lng': sender_lng,
        'recipient_lat': rec_lat, 
        'recipient_lng': rec_lng
    }
    origin(data, N)

[0.12666092459942477, 0.3502960300949628, 0.27937886580186305, -0.1619355795743852, -0.35482159152672804, -0.10335806669403383, 0.39430733566443016, -0.6516612866528423, -0.0581742049437175, -0.6991893213660602, -0.450609814604628, 0.7881601813905188, 0.6962343778413806, -0.8081396491427273, 0.13900362101532399, -0.1286532561351949, 0.6692510395671899, 0.25119848042185416, -0.45548946556687264, -0.43693402745718884, 0.6309792768005826, 0.4531413822330123, -0.4333078311814193, -0.7851576990182089, -0.8499673923773376, -0.4870599505051116, 0.04493938895486851, -0.5003560964481152, -0.38464404529061574, -0.3167718459728038, 0.8186910141425381, -0.5455446997425825, 0.3285225315920976, 0.16061226272047607, 0.3553807669049561, -0.7104447213320761, 0.016449671992623486, -0.002218149817892135, 0.139144702829568, 0.3196844188517203, 0.10617190668644363, -0.28597448566262257, -0.47723259569697446, 0.050152124300524153, 0.11952604623488605]
[-0.5280111014956872, -0.9365509415638711, -0.3727139145

In [2]:
np.random.seed(19)
embeddings = np.random.randn(10, 5)  # 示例数据
triu_i, triu_j = np.triu_indices(n = len(embeddings), k = 1)
embd_1 = embeddings[triu_i]
embd_2 = embeddings[triu_j]
embd_products = np.sum(embd_1 * embd_2, axis=1)
norm_1 = np.linalg.norm(embd_1, axis=1)
norm_2 = np.linalg.norm(embd_2, axis=1)
denominator = norm_1 * norm_2
epsilon = 1e-8
cos_sim = embd_products / (denominator + epsilon)
print(cos_sim)

[ 0.01980586 -0.04691727 -0.1658794  -0.86315881  0.13813405  0.31222567
 -0.24412446 -0.14098182 -0.82381543  0.0644523   0.39898939  0.22944097
  0.3955637   0.29523748  0.0614613  -0.41777082 -0.18695974  0.4069329
 -0.3307589   0.41413805 -0.37346038  0.43533739 -0.65749348  0.55861918
  0.09096204  0.95344771  0.20928519 -0.1933947  -0.82618261  0.23926067
 -0.18445639  0.10077178  0.21373598  0.33931499  0.44469047  0.28539734
 -0.26557542 -0.88212937  0.00309519  0.12979415  0.13096244 -0.58306605
  0.24435708  0.37212788 -0.16242179]


0.7703660422300194

In [37]:
from sklearn.metrics.pairwise import cosine_similarity
n = 10
for i in range(n - 1):
    for j in range(i + 1, 10):
        sk_sm = cosine_similarity([embeddings[i]], [embeddings[j]])[0][0]
        print(abs(sk_sm - cos_sim[int(i * n - ((i + 1) * i) / 2 + j - i - 1)]))
        # print(cos_sim[])
        # print(int(i * n - ((i + 1) * i) / 2 + j - i - 1))

3.3859762216259526e-10
1.10913889184161e-09
9.182252735939755e-10
8.087522873623243e-10
2.838299734131411e-11
5.212949516497645e-10
5.282423110042345e-10
1.1573660779751194e-10
5.475572217861213e-10
3.7604203084740107e-10
2.631746903736243e-11
1.9623874747409786e-10
5.4477602984270845e-11
1.0563285801623579e-09
2.5688257077050025e-09
6.07544486941336e-10
3.1449368109726095e-10
1.185871068010158e-09
4.199307568342192e-10
4.246866747159572e-11
5.96104415828691e-11
9.399383493757796e-10
2.0309894044334698e-10
9.755268814970464e-10
4.209824155942954e-10
2.708306356957735e-10
1.74012387832434e-10
6.03672278831624e-10
2.4864674208480153e-10
9.99452298700021e-10
1.801119531297246e-10
1.739363653108228e-10
4.97583918512845e-10
1.3717358846032646e-11
7.501599341708243e-10
2.561417189461679e-10
4.4993625492040223e-10
1.50672785359518e-09
7.936903356764446e-10
9.76500380556189e-10
2.9593824613094455e-10
4.396975006315529e-10
1.5645087447779815e-09
3.0179447829681294e-10
7.586170580609064e-10


In [17]:
testi, testj = np.triu_indices(n=20, k=1)
print(type(testi))
print(testi, testj)
print(len(testi))


<class 'numpy.ndarray'>
[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  1  1  1  1
  1  1  1  1  1  1  1  1  1  1  1  1  1  2  2  2  2  2  2  2  2  2  2  2
  2  2  2  2  2  2  3  3  3  3  3  3  3  3  3  3  3  3  3  3  3  3  4  4
  4  4  4  4  4  4  4  4  4  4  4  4  4  5  5  5  5  5  5  5  5  5  5  5
  5  5  5  6  6  6  6  6  6  6  6  6  6  6  6  6  7  7  7  7  7  7  7  7
  7  7  7  7  8  8  8  8  8  8  8  8  8  8  8  9  9  9  9  9  9  9  9  9
  9 10 10 10 10 10 10 10 10 10 11 11 11 11 11 11 11 11 12 12 12 12 12 12
 12 13 13 13 13 13 13 14 14 14 14 14 15 15 15 15 16 16 16 17 17 18] [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19  2  3  4  5  6
  7  8  9 10 11 12 13 14 15 16 17 18 19  3  4  5  6  7  8  9 10 11 12 13
 14 15 16 17 18 19  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19  5  6
  7  8  9 10 11 12 13 14 15 16 17 18 19  6  7  8  9 10 11 12 13 14 15 16
 17 18 19  7  8  9 10 11 12 13 14 15 16 17 18 19  8  9 10 11 12 13 14 15
 16 17 18 19  9 10 11 12 13 14 1

In [ ]:
# def test_haversine(lat1, lon1, lat2, lon2):
#     r = 6371
#     dlat = np.radians(lat2 - lat1)
#     dlon = np.radians(lon2 - lon1)
    
#     sin_dlat = np.sin(dlat / 2)
#     sin_dlon = np.sin(dlon / 2)
    
#     rad_lat1 = np.radians(lat1)
#     rad_lat2 = np.radians(lat2)
    
#     a = sin_dlat * sin_dlat + np.cos(rad_lat1) * np.cos(rad_lat2) * sin_dlon * sin_dlon
#     c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
#     d = r * c
#     return d 


# def test_share_degree_cal(s_lat1, s_lng1, r_lat1, r_lng1,
#                           s_lat2, s_lng2, r_lat2, r_lng2):
    
    
#     pass

# if __name__ == "__main__":
#     import time
#     start = time.time()
#     np.random.seed(2025)
#     lat1 = np.random.normal(0, 1, 32000000)
#     lon1 = np.random.normal(0, 1, 32000000)
#     lat2 = np.random.normal(0, 1, 32000000)
#     lon2 = np.random.normal(0, 1, 32000000)
#     print(f"Time for data: {time.time() - start}")
#     result = test_haversine(lat1, lon1, lat2, lon2)
#     print(f"Time for numpy: {time.time() - start}")
#     print(result[:10])

Time for data: 2.936812162399292
Time for numpy: 5.931141138076782
[176.16925591  82.71928231 450.28483736 101.20001897  60.13886381
  49.77139626 164.30682975 158.84220365 154.13782552 124.89655633]


In [ ]:
import numpy as np
def cal_dists(X, Y=None):
    lat1, lng1 = X[:, 0], X[:, 1]
    lat1, lng1 = (
        np.radians(lat1),
        np.radians(lng1),
    )
    if Y is None:
        lat2, lng2 = lat1, lng1
    else:
        lat2, lng2 = Y[:, 0], Y[:, 1]
        lat2, lng2 = np.radians(lat2), np.radians(lng2)
    # 用于配对矩阵的广播运算
    lat1 = np.expand_dims(lat1, axis=1)
    lng1 = np.expand_dims(lng1, axis=1)
    lat2 = np.expand_dims(lat2, axis=0)
    lng2 = np.expand_dims(lng2, axis=0)

    lat = lat2 - lat1
    lng = lng2 - lng1
    d = np.sin(lat * 0.5) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(lng * 0.5) ** 2
    return 2 * 6371.0088 * np.arcsin(np.sqrt(d))

if __name__ == "__main__":
    X = np.random.rand(, 2)
    Y = np.random.rand(3200000, 2)
    cal_dists(X)
    # res = cal_dists(X, Y)

In [7]:
from multiprocessing import Pool, cpu_count
from itertools import combinations
from sklearn.metrics.pairwise import cosine_similarity
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

def safe_cosine_similarity(a, b):
    if a is None or b is None or len(a) == 0 or len(b) == 0 or len(a) != len(b):
        return 0
    return cosine_similarity([a], [b])[0][0]

# 向量化Haversine距离计算
def haversine(lat1, lng1, lat2, lng2):
    lat1, lng1 = np.radians(lat1/1e6), np.radians(lng1/1e6)
    lat2, lng2 = np.radians(lat2/1e6), np.radians(lng2/1e6)
    dlat, dlng = lat2 - lat1, lng2 - lng1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlng/2)**2
    return 6371000 * 2 * np.arcsin(np.sqrt(a))

# 计算共享距离
def share_degree_calculation(flow_coordinates_1, flow_coordinates_2):
    if 0 in flow_coordinates_1 or 0 in flow_coordinates_2:
        return -1
    wb1_sender = (flow_coordinates_1[0], flow_coordinates_1[1])
    wb1_rec = (flow_coordinates_1[2], flow_coordinates_1[3])
    wb2_sender = (flow_coordinates_2[0], flow_coordinates_2[1])
    wb2_rec = (flow_coordinates_2[2], flow_coordinates_2[3])

    points = [wb1_sender, wb1_rec, wb2_sender, wb2_rec]
    orders = [[0, 1, 2, 3], [0, 2, 1, 3], [0, 2, 3, 1], [2, 0, 1, 3], [2, 0, 3, 1], [2, 3, 0, 1]]
    min_distance = float('inf')

    # 预先计算 Haversine 距离并存储到字典
    distance_cache = {}
    for i in range(len(points)):
        for j in range(i + 1, len(points)):
            
            point1 = points[i]
            point2 = points[j]
            distance_cache[tuple(sorted((point1, point2)))] = haversine(point1[0], point1[1], point2[0], point2[1])

    for order in orders:
        distance = 0
        for i in range(len(order) - 1):
            point1 = points[order[i]]
            point2 = points[order[i + 1]]
            # 从字典中查找预先计算的距离
            distance += distance_cache[tuple(sorted((point1, point2)))]
        min_distance = min(min_distance, distance)
    sep_distance = (distance_cache[tuple(sorted((wb1_sender, wb1_rec)))] +
                    distance_cache[tuple(sorted((wb2_sender, wb2_rec)))])
    
    return 1 - min_distance / max(sep_distance, 1e-9)

def parallel_process_group(args):
    """并行化建图主函数"""
    group_name, group_df = args
    
    # 预处理为数组格式提升性能
    data = {
        'ids': group_df['platform_order_id'].values,
        'sender_aois': group_df["sender_aoi_id"].values,
        'embeddings': group_df['embedding'].apply(np.array).values,
        'sender_lat': group_df['sender_lat'].values,
        'sender_lng': group_df['sender_lng'].values,
        'recipient_lat': group_df['recipient_lat'].values,
        'recipient_lng': group_df['recipient_lng'].values
    }
    
    # 生成候选对索引
    n = len(group_df)
    candidate_indices = [(i, j) for i in range(n) for j in range(i+1, n)]
    NUM_PROCESSES = max(cpu_count() - 2, 1)  # 动态调整进程数
    # 并行计算边
    with Pool(NUM_PROCESSES) as pool:
        chunk_size = max(len(candidate_indices) // (NUM_PROCESSES * 4), 100)
        edges = pool.starmap(
            compute_edge,
            [(i, j, data) for i, j in candidate_indices],
            chunksize=chunk_size
        )

    tabu_edges = set()
    # 构建图结构
    graph = nx.Graph()
    for edge in filter(None, edges):
        if isinstance(edge[2], str):
            tabu_edges.add(frozenset([edge[0], edge[1]]))
        else:
            graph.add_edge(edge[0], edge[1], similarity=edge[2])
    
    return (group_name, graph, tabu_edges)

def compute_edge(i, j, data):
    """单条边的计算任务"""
    # 条件过滤：同发送区域
    if data['sender_aois'][i] != data['sender_aois'][j]:
        return None
    
    # 计算相似度
    emb_i = data['embeddings'][i]
    emb_j = data['embeddings'][j]
    sim = safe_cosine_similarity(emb_i, emb_j)
    
    # 计算顺路度
    coords_i = (
        data['sender_lat'][i], data['sender_lng'][i],
        data['recipient_lat'][i], data['recipient_lng'][i]
    )
    coords_j = (
        data['sender_lat'][j], data['sender_lng'][j],
        data['recipient_lat'][j], data['recipient_lng'][j]
    )
    share_deg = share_degree_calculation(coords_i, coords_j)
    
    # 边条件判断
    if sim >= cos_threshold or share_deg >= share_degree_threshold:
        return (
            data['ids'][i], 
            data['ids'][j], 
            sim + 2 * share_deg
        )
    elif sim < tabu_cosine_sim_threshold and share_deg < tabu_share_degree_threshold:
        # Record the forbidden combination (i, j)
        return (
            data['ids'][i],
            data['ids'][j], 
            "forbidden"
        )
    return None

def construct_graph_parallel(raw_graph_dict):
    """并行构图主函数, 更新禁忌表
    """
    graph_dict = dict() # 记录网络结构
    graph_tabu_edges = dict() # 记录该网络中不能被连接的边
    # with ThreadPoolExecutor(max_workers=128) as executor:
    #     futures = []
        # for group, graph in raw_graph_dict.items():
        #     futures.append(executor.submit(parallel_process_group, (group, graph)))
        # for future in tqdm(futures, desc="Processing cliques", unit="clique"):
        #     graph_name, graph = future.result() 
        #     graph_dict[graph_name] = graph
    for group, graph in raw_graph_dict.items():
    #     futures.append(executor.submit(parallel_process_group, (group, graph)))
    # for future in tqdm(futures, desc="Processing cliques", unit="clique"):
        graph_name, graph, tabu_edges = parallel_process_group((group, graph))
        graph_dict[graph_name] = graph
        graph_tabu_edges[graph_name] = tabu_edges
        print(f"Complete graph {graph_name}, len of tabu_edges: {len(tabu_edges)}")

    return graph_dict, graph_tabu_edges

if __name__ == "__main__":
    # data = pd.read_parquet("./data/data_beijing_clique_big.parquet")
    from collections import Counter
    raw_data = pd.read_parquet("./data/clique_test_0401.parquet")
    # Kunshan 42 network data.
    start = time.time()
    raw_graph_dict = dict()
    for graph_name in Counter(raw_data['network_id']).keys():
        raw_graph_dict[str(graph_name)] = raw_data[raw_data['network_id'] == graph_name]

    graph_dict, graph_tabu_edges = construct_graph_parallel(raw_graph_dict)
    print(f"Improved Time Consumption: {time.time()-start}")
    # display(clique_df.head())

(650, 2)


In [4]:
res

array([[ 62.67350527,  62.69323429,  18.29362962, ...,  71.1675384 ,
         21.66247079,  38.66021417],
       [ 12.2640567 ,   4.88251418,  75.46170434, ...,  12.94493346,
         56.48176011,  86.72961652],
       [ 24.3189937 ,  30.18961167,  65.02575085, ...,  42.46868568,
         55.67849293,  62.05739041],
       ...,
       [ 76.86720001,  84.84975065,  96.34965623, ...,  96.79140709,
        100.66214679,  64.23611344],
       [ 18.14597484,  24.51327029,  96.07257183, ...,  30.91778391,
         81.44058084,  95.44348812],
       [ 58.81868756,  57.17313414,  22.802146  , ...,  63.98921978,
         10.18411155,  51.32092737]])

In [5]:
distances[:20]

array([217.99454356, 161.09643416, 337.93776656,  85.99167492,
       185.83304345, 284.37905471, 350.14311068, 305.22501717,
       321.1614812 , 316.56406765, 161.46227781, 106.74612841,
       280.26951165, 274.60290229, 515.49327112,  87.35301208,
       215.5845845 , 184.77337991,  63.51320691, 342.2247437 ])

In [ ]:
# 向量化Haversine距离计算
def haversine(lat1, lng1, lat2, lng2):
    lat1, lng1 = np.radians(lat1/1e6), np.radians(lng1/1e6)
    lat2, lng2 = np.radians(lat2/1e6), np.radians(lng2/1e6)
    dlat, dlng = lat2 - lat1, lng2 - lng1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlng/2)**2
    return 6371000 * 2 * np.arcsin(np.sqrt(a))

# 计算共享距离
def share_degree_calculation(flow_coordinates_1, flow_coordinates_2):
    if 0 in flow_coordinates_1 or 0 in flow_coordinates_2:
        return -1
    wb1_sender = (flow_coordinates_1[0], flow_coordinates_1[1])
    wb1_rec = (flow_coordinates_1[2], flow_coordinates_1[3])
    wb2_sender = (flow_coordinates_2[0], flow_coordinates_2[1])
    wb2_rec = (flow_coordinates_2[2], flow_coordinates_2[3])

    points = [wb1_sender, wb1_rec, wb2_sender, wb2_rec]
    orders = [[0, 1, 2, 3], [0, 2, 1, 3], [0, 2, 3, 1], [2, 0, 1, 3], [2, 0, 3, 1], [2, 3, 0, 1]]
    min_distance = float('inf')

    # 预先计算 Haversine 距离并存储到字典
    distance_cache = {}
    for i in range(len(points)):
        for j in range(i + 1, len(points)):
            
            point1 = points[i]
            point2 = points[j]
            distance_cache[tuple(sorted((point1, point2)))] = haversine(point1[0], point1[1], point2[0], point2[1])

    for order in orders:
        distance = 0
        for i in range(len(order) - 1):
            point1 = points[order[i]]
            point2 = points[order[i + 1]]
            # 从字典中查找预先计算的距离
            distance += distance_cache[tuple(sorted((point1, point2)))]
        min_distance = min(min_distance, distance)

    sep_distance = (distance_cache[tuple(sorted((wb1_sender, wb1_rec)))] +
                    distance_cache[tuple(sorted((wb2_sender, wb2_rec)))])
    
    return 1 - min_distance / max(sep_distance, 1e-9)

In [3]:
import time 
start = time.time()
for i in range(8000):
    for j in range(8000):
        continue 
print(time.time() - start)

1.2547352313995361
